In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import pandas as pd
from tqdm import tqdm

In [ ]:
with open("../data/fold_list.pkl", "rb") as f:
    fold_list = pickle.load(f)

In [ ]:
tempo_df = pd.DataFrame(columns=["tempo", "fold"])
length = 0

for fid, fold in enumerate(fold_list):
    for cluster in fold:
        for prot in cluster:
            for nid in prot[2]:
                length += 1

progress = tqdm(total=length, desc="Processing tempo data")
for fid, fold in enumerate(fold_list):
    for cluster in fold:
        for prot in cluster:
            tempo_list = []
            for nid in prot[2]:
                progress.update(1)
                with open(f"../data/NCBI_pkl/NCBI_{nid}.pkl", "rb") as f:
                    tempo = pickle.load(f)["tempo"]
                    tempo_list.append(tempo)
            tempo_list = np.array(tempo_list)
            tempo = np.mean(tempo_list, axis=0)
            tempo_df = pd.concat([tempo_df, pd.DataFrame({"tempo": tempo, "fold": np.array([fid]*len(tempo))})], ignore_index=True)

In [ ]:
kde_df = pd.DataFrame(columns=["x"] + [f"fold_{i}" for i in range(len(fold_list))] + ["All"])
x = np.linspace(np.floor(min(tempo_df["tempo"].values)), np.ceil(max(tempo_df["tempo"].values)), 1000)
kde_df["x"] = x
for fold in range(len(fold_list)):
    fold_tempo = tempo_df[tempo_df["fold"] == fold]["tempo"].values
    fold_tempo = np.array(fold_tempo,dtype=float)
    density = gaussian_kde(fold_tempo)
    y = density(x)
    kde_df[f"fold_{fold}"] = y
tempo = tempo_df["tempo"].values
tempo = np.array(tempo, dtype=float)
density = gaussian_kde(tempo)
y = density(x)
kde_df["All"] = y

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ylim = 1
for col in kde_df.columns.to_list()[1:]:
    ax.plot(kde_df["x"], kde_df[col], label=f"Fold {col.split('_')[-1]}" if col != "All" else "All", linewidth=2)
for percent, color in zip([50, 90, 95],["gray", "blue", "red"]):
    x_val = np.percentile(tempo_df["tempo"].values, percent)
    ax.axvline(x=x_val, color=color, linestyle="--")
    y_pos = ylim * 0.95
    ax.text(x_val, y_pos, f"{percent}%", 
            color=color, fontsize=10, 
            ha='center', va='top',
            bbox=dict(facecolor='white', edgecolor=color, alpha=0.8))
    print(f"{percent}th percentile: {x_val}")
ax.legend()
ax.set_xlabel("Translation Tempo")
ax.set_ylabel("KDE")
ax.set_ylim(0, ylim)
# ax.set_title("Distribution of Translation Tempo across folds")
fig.tight_layout()
plt.savefig("Tempo_distribution.png", dpi=300)